# EmberRealm sprite LoRA — free Colab

Trains a style LoRA on the game's own 2,934 sprites, then turns what it generates back into
real sprites.

**Nothing to upload.** The art lives in the repo, so this clones it and rebuilds the training
set here — which also means it always matches whatever is in `main`.

**First: turn the GPU on.** Runtime → Change runtime type → T4 GPU. Without it the next cell
tells you and nothing else will work.

Free Colab disconnects idle sessions and has a daily GPU allowance, so the run is sized to
finish in well under an hour and saves a checkpoint each epoch rather than only at the end.

In [ ]:
#@title 1. Check the GPU  { display-mode: "form" }
import subprocess, sys
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
if out.returncode != 0 or not out.stdout.strip():
    sys.exit('No GPU. Runtime -> Change runtime type -> T4 GPU, then run this again.')
print(out.stdout.strip())
print('\nA T4 has 16 GB, which is four times the card this was written for -- so this notebook\n'
      'uses a larger batch and skips the memory tricks the 4 GB config needs.')

In [ ]:
#@title 2. Get the art and build the training set  { display-mode: "form" }
subset = "weapons" #@param ["weapons", "everything"]

import os, subprocess, shutil
if not os.path.isdir('/content/emberrealm'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Setijinn/emberrealm.git', '/content/emberrealm'], check=True)
os.chdir('/content/emberrealm')

cmd = [sys.executable if False else 'python', 'tools/dataset.py']
if subset != 'everything':
    cmd += ['--only', subset]
print(subprocess.run(cmd, capture_output=True, text=True).stdout)

# sd-scripts reads the repeat count from the folder name: images x repeats x epochs = steps.
# Aim for 2,000-6,000. Fewer images therefore need more repeats.
n = len([f for f in os.listdir('_dataset') if f.endswith('.png')])
repeats = max(1, round(3000 / max(n, 1) / 2))
dst = f'/content/train/{repeats}_emberrealm'
shutil.rmtree('/content/train', ignore_errors=True)
os.makedirs(dst, exist_ok=True)
for f in os.listdir('_dataset'):
    shutil.copy(f'_dataset/{f}', dst)
print(f'{n} images, {repeats} repeats, 2 epochs = {n*repeats*2} steps')
print('a T4 does roughly 0.28 s/step at 320px, so about '
      f'{n*repeats*2*0.28/60:.0f} minutes')

In [ ]:
#@title 3. Install the trainer (~3 min)  { display-mode: "form" }
%cd /content
![ -d sd-scripts ] || git clone --depth 1 https://github.com/kohya-ss/sd-scripts
%cd /content/sd-scripts
!pip -q install -r requirements.txt 2>&1 | tail -2
!pip -q install bitsandbytes xformers accelerate 2>&1 | tail -2
# non-interactive accelerate config: one GPU, fp16, no distributed anything
import os
os.makedirs('/root/.cache/huggingface/accelerate', exist_ok=True)
open('/root/.cache/huggingface/accelerate/default_config.yaml', 'w').write(
    'compute_environment: LOCAL_MACHINE\ndistributed_type: NO\nmixed_precision: fp16\n'
    'num_processes: 1\nuse_cpu: false\n')
print('ready')

In [ ]:
#@title 4. Train  { display-mode: "form" }
# A T4 config, NOT the 4 GB one in tools/train/lora_4gb.toml. With 16 GB there is no need for
# gradient checkpointing or an 8-bit optimiser, and a real batch is both faster and steadier.
cfg = '''
[model_arguments]
pretrained_model_name_or_path = "runwayml/stable-diffusion-v1-5"

[dataset_arguments]
train_data_dir = "/content/train"
resolution = "320,320"
enable_bucket = false
caption_extension = ".txt"
shuffle_caption = false
keep_tokens = 2

[training_arguments]
output_dir = "/content/lora"
output_name = "emberrealm"
save_model_as = "safetensors"
max_train_epochs = 2
train_batch_size = 4
mixed_precision = "fp16"
save_precision = "fp16"
cache_latents = true
xformers = true
seed = 7
save_every_n_epochs = 1

[optimizer_arguments]
optimizer_type = "AdamW8bit"
learning_rate = 1e-4
unet_lr = 1e-4
text_encoder_lr = 5e-5
lr_scheduler = "cosine_with_restarts"
lr_warmup_steps = 100

[network_arguments]
network_module = "networks.lora"
network_dim = 16
network_alpha = 8
'''
open('/content/t4.toml', 'w').write(cfg)
!accelerate launch --num_cpu_threads_per_process 2 train_network.py --config_file /content/t4.toml

In [ ]:
#@title 5. Generate some sprites  { display-mode: "form" }
prompt = "emberrealm, pixel art sprite, items weapons, sword, upright, flat violet background" #@param {type:"string"}
count = 8 #@param {type:"slider", min:1, max:24, step:1}

!pip -q install diffusers transformers safetensors 2>&1 | tail -1
import torch, os
from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained('runwayml/stable-diffusion-v1-5',
                                               torch_dtype=torch.float16,
                                               safety_checker=None).to('cuda')
pipe.load_lora_weights('/content/lora', weight_name='emberrealm.safetensors')
os.makedirs('/content/out', exist_ok=True)
imgs = pipe([prompt] * count, num_inference_steps=28, guidance_scale=7.0,
            height=320, width=320).images
for i, im in enumerate(imgs):
    im.save(f'/content/out/{i:03d}.png')

from PIL import Image
sheet = Image.new('RGB', (320 * len(imgs), 320))
for i, im in enumerate(imgs):
    sheet.paste(im, (i * 320, 0))
sheet.resize((min(1600, 160 * len(imgs)), int(320 * min(1600, 160*len(imgs)) / (320*len(imgs)))))

In [ ]:
#@title 6. Turn them into actual sprites  { display-mode: "form" }
# What the model emits is a 320px painting OF a sprite: soft edges, thousands of colours, no alpha.
# spritify keys the background, downscales by mode, snaps to the category's real palette, hardens
# the alpha and keeps the largest connected piece -- then scores it against the real art.
category = "Items \u00b7 weapons" #@param {type:"string"}
size = 64 #@param {type:"slider", min:16, max:128, step:8}

%cd /content/emberrealm
import glob, subprocess
os.makedirs('/content/sprites', exist_ok=True)
for f in sorted(glob.glob('/content/out/*.png')):
    dst = '/content/sprites/' + os.path.basename(f)
    r = subprocess.run(['python', 'tools/spritify.py', f, '--cat', category,
                        '--size', str(size), '--out', dst], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip()[:200])

from PIL import Image
sp = [Image.open(f).convert('RGBA') for f in sorted(glob.glob('/content/sprites/*.png'))]
if sp:
    Z = 4
    W = sum(i.width for i in sp) * Z + 10 * len(sp)
    H = max(i.height for i in sp) * Z + 10
    sheet = Image.new('RGBA', (W, H), (22, 19, 26, 255))
    x = 5
    for i in sp:
        z = i.resize((i.width * Z, i.height * Z), Image.NEAREST)
        sheet.alpha_composite(z, (x, 5)); x += z.width + 10
    display(sheet)
print('\nlower style scores are more like the real art; anything over ~2 is worth rejecting')

In [ ]:
#@title 7. Download the LoRA and the sprites  { display-mode: "form" }
# Do this before the session times out -- free Colab does not keep anything.
import shutil
from google.colab import files
shutil.make_archive('/content/emberrealm_lora', 'zip', '/content/lora')
shutil.make_archive('/content/emberrealm_sprites', 'zip', '/content/sprites')
files.download('/content/emberrealm_lora.zip')
files.download('/content/emberrealm_sprites.zip')

## If it goes wrong

- **`No GPU`** — Runtime → Change runtime type → T4 GPU.
- **Everything it makes looks like a photograph** — the trigger word is missing. Every prompt has
  to start `emberrealm, pixel art sprite, ...` because that is how the captions were written.
- **It hands back sprites you already own** — overtrained. Use the epoch-1 checkpoint instead of
  epoch 2; both are in the zip.
- **Blurry, or the shapes are mush** — that is the base model showing through, meaning undertrained.
  Raise the repeats in cell 2 for another ~1,500 steps.
- **Everything comes out diagonal** — say `upright` or `horizontal` in the prompt. The corpus is
  962 upright, 242 diagonal, 36 horizontal, and orientation is a caption token precisely so it is a
  dial rather than a habit.